<a href="https://colab.research.google.com/github/muhammad-bin-nasir/Sentiment-Analysis/blob/main/SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define your project directory paths
# IMPORTANT: Change 'Your_Project_Folder' to your actual folder name
base_path = '/content/drive/My Drive/Sentiment analysis'
audio_folder = os.path.join(base_path, 'Audio')
labels_csv = os.path.join(base_path, 'labels_fixed.csv')

# Create Output/MFCC directory if it doesn't exist
output_dir = os.path.join(base_path, 'Output', 'MFCC')
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Output will be saved to: {output_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Output will be saved to: /content/drive/My Drive/Sentiment analysis/Output/MFCC


<h1> MFCC

.npy files for traininng

In [8]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm # Import tqdm for progress tracking

def extract_features(audio_path, n_mfcc=173, n_mels=200):
    """
    Extracts MFCC features from an audio file.
    """
    try:
        y, sr = librosa.load(audio_path, sr=None)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_mels=n_mels)
        mfcc_mean = np.mean(mfcc, axis=1)
        return mfcc_mean
    except Exception as e:
        # We'll print errors specifically since they are critical
        print(f"\n❌ Error processing {os.path.basename(audio_path)}: {e}")
        return None

def make_features(audio_folder, labels_csv, output_path):
    """
    Processes audio, tracks progress with a bar, and saves to Drive.
    """
    # Load dataset
    df = pd.read_csv(labels_csv)
    features, labels = [], []

    # Define save names inside the output directory
    x_file = os.path.join(output_path, "features_MFCC.npy")
    y_file = os.path.join(output_path, "labels_MFCC.npy")

    # Initialize tqdm progress bar
    # 'unit' defines the label for the counter
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Extracting Audio Features", unit="file"):

        fname = row["filename"]
        label = row["label"]
        audio_path = os.path.join(audio_folder, fname)

        if not os.path.exists(audio_path):
            continue

        feat = extract_features(audio_path, n_mfcc=173, n_mels=200)

        if feat is not None:
            features.append(feat)
            labels.append(label)

    # Convert to arrays
    X = np.array(features)
    y = np.array(labels)

    # Save directly to the Drive output folder
    np.save(x_file, X)
    np.save(y_file, y)

    print(f"\n🎉 Success! Processed {X.shape[0]} samples.")
    print(f"📁 Files saved in: {output_path}")

# --- Execute ---
make_features(audio_folder, labels_csv, output_dir)

Extracting Audio Features:   0%|          | 0/12438 [00:00<?, ?file/s]


🎉 Success! Processed 12438 samples.
📁 Files saved in: /content/drive/My Drive/Sentiment analysis/Output/MFCC


Verification

In [9]:
import numpy as np
import os

# Define the paths to your saved files
x_file = os.path.join(output_dir, "features_MFCC.npy")
y_file = os.path.join(output_dir, "labels_MFCC.npy")

# Load the data
X_loaded = np.load(x_file)
y_loaded = np.load(y_file)

# Check shapes
print(f"📊 Feature matrix shape: {X_loaded.shape}") # Expected: (samples, 173)
print(f"📊 Labels array shape: {y_loaded.shape}")    # Expected: (samples,)
print(f"✅ Data verified and ready for training.")

📊 Feature matrix shape: (12438, 173)
📊 Labels array shape: (12438,)
✅ Data verified and ready for training.
